# R1: Multi-Seed Evaluation — RSQwen & Qwen Vanilla

**Purpose:** Run 3 seeds (42, 123, 456) for both RSQwen (Stage 1 + Stage 2) and Qwen Vanilla (Stage 1 only) to produce mean ± std metrics.

**Output:** `results_multiseed.json` + `all_predictions.json`

**Estimated time:** 10-14h on A100, 14-18h on V100

**IMPORTANT:** Run all cells sequentially. Auto-saves after each seed.

## Step 1: Install Dependencies

In [ ]:
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.0
!pip install -q transformers datasets accelerate sentencepiece
!pip install -q scikit-learn pandas numpy tqdm openpyxl
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes
!pip install -q peft

import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GRPC_VERBOSITY'] = 'ERROR'
warnings.filterwarnings('ignore')
print('Dependencies installed')

## Step 2: Mount Google Drive & Setup Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, sys, os
from pathlib import Path

# === CONFIGURE THIS ===
DRIVE_BASE = Path('/content/drive/MyDrive/ViTHSD')
# ======================

WORK_DIR = Path('/content/vithsd')
WORK_DIR.mkdir(exist_ok=True)

# Copy source files to working directory
for f in ['config.py', 'data_preparation.py', 'evaluation.py', 'models.py']:
  src = DRIVE_BASE / 'src' / f
  dst = WORK_DIR / f
  if src.exists():
    shutil.copy2(src, dst)
    print(f' Copied {f}')
  else:
    raise FileNotFoundError(f'Missing: {src}')

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))

OUTPUT_DIR = DRIVE_BASE / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Working dir: {WORK_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

## Step 3: Patch config.py paths for Colab

In [ ]:
import config
config.DATA_DIR = DRIVE_BASE / 'data'
config.OUTPUT_DIR = OUTPUT_DIR
config.BASE_DIR = WORK_DIR

for f in ['train.xlsx', 'test.xlsx']:
  p = config.DATA_DIR / f
  assert p.exists(), f'Missing data file: {p}'
  print(f' Found: {f}')

print('Config patched for Colab')

## Step 4: GPU Detection

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
  raise RuntimeError('No GPU! Go to Runtime > Change runtime type > GPU')

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

if gpu_mem >= 35:
  BATCH_SIZE = 8
elif gpu_mem >= 14:
  BATCH_SIZE = 4
else:
  BATCH_SIZE = 2

print(f'Batch size: {BATCH_SIZE}')

## Step 5: Import Modules & Load Data

In [ ]:
from config import FINAL_LABELS
from data_preparation import load_dataset_A
from evaluation import Evaluator
from models import create_model

train_texts, train_labels = load_dataset_A('train', data_dir=config.DATA_DIR)
test_texts, test_labels = load_dataset_A('test', data_dir=config.DATA_DIR)

print(f'Train: {len(train_texts)} samples')
print(f'Test: {len(test_texts)} samples')
print(f'Labels: {len(FINAL_LABELS)}')

## Step 6: Load Rationale Data (for RSQwen Stage 2)

In [ ]:
import json

rationale_path = DRIVE_BASE / 'data' / 'dataset_rationale.json'
assert rationale_path.exists(), f'Missing: {rationale_path}'

with open(rationale_path, 'r', encoding='utf-8') as f:
  rationale_data = json.load(f)

r_train_texts, r_train_implied, r_train_rationale, r_train_labels = [], [], [], []
for r in rationale_data:
  if str(r.get('dataset', '')).lower().strip() != 'train':
    continue
  content = (r.get('content') or '').strip()
  implied = (r.get('implied_statement') or '').strip()
  if not content or not implied:
    continue
  r_train_texts.append(content)
  r_train_implied.append(implied)
  r_train_rationale.append(r.get('rationale', []))
  r_train_labels.append(r.get('labels', []))

alignment_pairs = [{'content': c, 'implied': i} for c, i in zip(r_train_texts, r_train_implied)]

print(f'Rationale train samples: {len(r_train_texts)}')
print(f'Alignment pairs: {len(alignment_pairs)}')

## Step 7: Define Seed & Stage 2 Training Functions

In [ ]:
import random
import numpy as np
import torch
import gc
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  print(f' Seed set to {seed}')


def train_stage2(model, pairs, rationale_list, labels_list, num_epochs=1, learning_rate=5e-5):
  """
  Stage 2: Semantic alignment using rationale-augmented continuation training.
  Continues training the LoRA adapter with CoT-formatted data.
  """
  print(f' Stage 2: {len(pairs)} pairs, {num_epochs} epoch(s), lr={learning_rate}')

  training_texts = []
  for i, pair in enumerate(pairs):
    content = pair['content']
    implied = pair['implied']
    rat = rationale_list[i] if i < len(rationale_list) else []
    lab = labels_list[i] if i < len(labels_list) else ['normal']
    input_text = model._format_input_cot(content)
    output_text = model._format_output_cot(lab, rat, implied)
    training_texts.append(input_text + output_text)

  class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
      self.encodings = tokenizer(
        texts, truncation=True, padding=True,
        max_length=max_length, return_tensors='pt'
      )
    def __len__(self):
      return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
      return {
        'input_ids': self.encodings['input_ids'][idx],
        'attention_mask': self.encodings['attention_mask'][idx]
      }

  dataset = TextDataset(training_texts, model.tokenizer, model.max_length)
  dataloader = DataLoader(dataset, batch_size=model.batch_size, shuffle=True)
  optimizer = torch.optim.AdamW(model.model.parameters(), lr=learning_rate)

  model.model.train()
  for epoch in range(num_epochs):
    total_loss = 0
    progress = tqdm(dataloader, desc=f' Stage2 Epoch {epoch+1}/{num_epochs}')
    for batch in progress:
      input_ids = batch['input_ids'].to(model.device)
      attention_mask = batch['attention_mask'].to(model.device)
      optimizer.zero_grad()
      outputs = model.model(
        input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
      )
      loss = outputs.loss
      total_loss += loss.item()
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.model.parameters(), 1.0)
      optimizer.step()
      progress.set_postfix({'loss': f'{loss.item():.4f}'})
    avg_loss = total_loss / len(dataloader)
    print(f' Stage2 Epoch {epoch+1}: avg_loss = {avg_loss:.4f}')
  print(' Stage 2 complete')


def cleanup_model(model):
  if hasattr(model, 'model') and model.model is not None:
    del model.model
  if hasattr(model, 'tokenizer') and model.tokenizer is not None:
    del model.tokenizer
  del model
  gc.collect()
  torch.cuda.empty_cache()
  print(' GPU memory freed')


def convert_to_serializable(obj):
  if isinstance(obj, dict):
    return {k: convert_to_serializable(v) for k, v in obj.items()}
  elif isinstance(obj, (list, tuple)):
    return [convert_to_serializable(item) for item in obj]
  elif isinstance(obj, (np.integer, np.floating)):
    return float(obj)
  elif isinstance(obj, np.ndarray):
    return obj.tolist()
  return obj

print('Helper functions defined')

## Step 8: Configuration

If Colab disconnects mid-run, change `START_FROM_INDEX` to skip completed runs.
- `0` = start from beginning
- `2` = skip seed 42 (both models)
- `4` = skip seeds 42 and 123

In [ ]:
SEEDS = [42, 123, 456]
START_FROM_INDEX = 0 # Change this to resume after disconnect

all_results = {'rsqwen': {}, 'vanilla': {}}
all_predictions = {'rsqwen': {}, 'vanilla': {}}

# Try to load partial results from previous runs
partial_path = OUTPUT_DIR / 'results_multiseed_partial.json'
partial_pred_path = OUTPUT_DIR / 'all_predictions_partial.json'
if partial_path.exists() and START_FROM_INDEX > 0:
  with open(partial_path, 'r') as f:
    all_results = json.load(f)
  print(f'Loaded partial results: RSQwen seeds={list(all_results["rsqwen"].keys())}')
if partial_pred_path.exists() and START_FROM_INDEX > 0:
  with open(partial_pred_path, 'r') as f:
    all_predictions = json.load(f)
  print(f'Loaded partial predictions')

print(f'Seeds: {SEEDS}, Starting from index: {START_FROM_INDEX}')

## Step 9: Run All Multi-Seed Experiments

For each seed:
1. **RSQwen**: Stage 1 (2 epochs on ~7540 samples) -> Stage 2 (1 epoch on ~1221 pairs) -> Predict
2. **Vanilla**: Stage 1 only (2 epochs) -> Predict

Auto-saves after each seed to Google Drive.

In [ ]:
import time

total_runs = len(SEEDS) * 2

for seed_idx, seed in enumerate(SEEDS):
  # === RSQwen (Stage 1 + Stage 2) ===
  run_idx = seed_idx * 2
  if run_idx < START_FROM_INDEX:
    print(f'\nSkipping RSQwen seed={seed} (already done)')
  else:
    print(f'\n{"="*70}')
    print(f'RUN {run_idx+1}/{total_runs}: RSQwen seed={seed}')
    print(f'{"="*70}')
    t0 = time.time()

    set_seed(seed)
    evaluator = Evaluator()
    model_rs = create_model(
      'qwen', dataset_type='A',
      batch_size=BATCH_SIZE, num_epochs=2,
      learning_rate=2e-4, lora_r=8, lora_alpha=16
    )

    print(' Stage 1: Training on Dataset A...')
    set_seed(seed)
    model_rs.train(train_texts, train_labels)

    print(' Stage 2: Rationale alignment...')
    set_seed(seed)
    train_stage2(model_rs, alignment_pairs, r_train_rationale, r_train_labels,
           num_epochs=1, learning_rate=5e-5)

    print(' Predicting...')
    pred_rs, raw_rs = model_rs.predict(test_texts)
    res_rs = evaluator.evaluate(test_labels, pred_rs, 'RSQwen', f'seed_{seed}')

    elapsed = (time.time() - t0) / 60
    print(f' DONE in {elapsed:.1f} min | F1-Micro={res_rs["f1_micro"]:.4f}, F1-Macro={res_rs["f1_macro"]:.4f}')

    all_results['rsqwen'][str(seed)] = convert_to_serializable(res_rs)
    all_predictions['rsqwen'][str(seed)] = pred_rs
    cleanup_model(model_rs)

  # === Vanilla (Stage 1 only) ===
  run_idx = seed_idx * 2 + 1
  if run_idx < START_FROM_INDEX:
    print(f'\nSkipping Vanilla seed={seed} (already done)')
  else:
    print(f'\n{"="*70}')
    print(f'RUN {run_idx+1}/{total_runs}: Vanilla seed={seed}')
    print(f'{"="*70}')
    t0 = time.time()

    set_seed(seed)
    evaluator = Evaluator()
    model_van = create_model(
      'qwen', dataset_type='A',
      batch_size=BATCH_SIZE, num_epochs=2,
      learning_rate=2e-4, lora_r=8, lora_alpha=16
    )

    print(' Stage 1: Training on Dataset A...')
    set_seed(seed)
    model_van.train(train_texts, train_labels)

    print(' Predicting...')
    pred_van, raw_van = model_van.predict(test_texts)
    res_van = evaluator.evaluate(test_labels, pred_van, 'Vanilla', f'seed_{seed}')

    elapsed = (time.time() - t0) / 60
    print(f' DONE in {elapsed:.1f} min | F1-Micro={res_van["f1_micro"]:.4f}, F1-Macro={res_van["f1_macro"]:.4f}')

    all_results['vanilla'][str(seed)] = convert_to_serializable(res_van)
    all_predictions['vanilla'][str(seed)] = pred_van
    cleanup_model(model_van)

  # Auto-save to Drive after each seed
  with open(OUTPUT_DIR / 'results_multiseed_partial.json', 'w') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
  with open(OUTPUT_DIR / 'all_predictions_partial.json', 'w') as f:
    json.dump(convert_to_serializable(all_predictions), f, indent=2, ensure_ascii=False)
  print(f' Partial results saved to Drive (seed {seed} done)')

print(f'\n{"="*70}')
print('ALL MULTI-SEED RUNS COMPLETE')
print(f'{"="*70}')

## Step 10: Aggregate Results (Mean +/- Std)

In [ ]:
import numpy as np

def aggregate_results(results_dict, model_name):
  seeds = sorted(results_dict.keys())
  if not seeds:
    return None

  metrics = ['f1_micro', 'f1_macro', 'precision_micro', 'precision_macro',
        'recall_micro', 'recall_macro', 'subset_accuracy', 'hamming_loss']

  agg = {'model': model_name, 'seeds': seeds, 'n_seeds': len(seeds)}

  for m in metrics:
    vals = [results_dict[s][m] for s in seeds]
    agg[m] = {
      'mean': float(np.mean(vals)),
      'std': float(np.std(vals)),
      'per_seed': {s: v for s, v in zip(seeds, vals)}
    }

  label_names = list(results_dict[seeds[0]]['f1_per_label'].keys())
  agg['f1_per_label'] = {}
  for label in label_names:
    vals = [results_dict[s]['f1_per_label'][label] for s in seeds]
    agg['f1_per_label'][label] = {
      'mean': float(np.mean(vals)),
      'std': float(np.std(vals)),
      'per_seed': {s: v for s, v in zip(seeds, vals)}
    }

  return agg

agg_rsqwen = aggregate_results(all_results['rsqwen'], 'RSQwen')
agg_vanilla = aggregate_results(all_results['vanilla'], 'Vanilla')

print(f'{"="*70}')
print(f'{"MULTI-SEED RESULTS SUMMARY":^70}')
print(f'{"="*70}')
print(f'{"":>25} {"RSQwen":>20} {"Vanilla":>20}')
print(f'{"-"*65}')

for m in ['f1_micro', 'f1_macro', 'precision_micro', 'recall_micro', 'subset_accuracy']:
  rs = agg_rsqwen[m]
  va = agg_vanilla[m]
  print(f'{m:>25} {rs["mean"]*100:6.2f} +/- {rs["std"]*100:.2f}  {va["mean"]*100:6.2f} +/- {va["std"]*100:.2f}')

print(f'\n{"Per-label F1 (mean %)":>25}')
print(f'{"-"*65}')
for label in FINAL_LABELS:
  rs_l = agg_rsqwen['f1_per_label'][label]
  va_l = agg_vanilla['f1_per_label'][label]
  print(f'{label:>25} {rs_l["mean"]*100:6.2f} +/- {rs_l["std"]*100:.2f}  {va_l["mean"]*100:6.2f} +/- {va_l["std"]*100:.2f}')

## Step 11: Save Final Results

In [ ]:
final_output = {
  'rsqwen': {
    'aggregated': convert_to_serializable(agg_rsqwen),
    'per_seed': convert_to_serializable(all_results['rsqwen'])
  },
  'vanilla': {
    'aggregated': convert_to_serializable(agg_vanilla),
    'per_seed': convert_to_serializable(all_results['vanilla'])
  }
}

with open(OUTPUT_DIR / 'results_multiseed.json', 'w') as f:
  json.dump(final_output, f, indent=2, ensure_ascii=False)
print(f'Results saved: {OUTPUT_DIR / "results_multiseed.json"}')

with open(OUTPUT_DIR / 'all_predictions.json', 'w') as f:
  json.dump(convert_to_serializable(all_predictions), f, indent=2, ensure_ascii=False)
print(f'Predictions saved: {OUTPUT_DIR / "all_predictions.json"}')

with open(OUTPUT_DIR / 'test_labels.json', 'w') as f:
  json.dump(test_labels, f, indent=2, ensure_ascii=False)
print(f'Ground truth saved: {OUTPUT_DIR / "test_labels.json"}')

print('\nDONE! Download results_multiseed.json and all_predictions.json from Drive.')
print('Then run R3_R6_R8_analysis.ipynb for statistical tests.')